In [0]:
# Gold layer: fact_web_sessions (Kimball periodic-grain fact - web
#         sessions) - gold enrichment #3 (sessionization, MUST-DECIDE
#         #6; 30-minute inactivity gap CONFIRMED by the user).
# Source: b2b_ecommerce.silver.logs - every web event, customer_sk-
#         resolved via the same auth_user = dim_customers.email
#         bridge as fact_web_logs (SCD1, unique email, lossless LEFT
#         join).
# Session definition (user-confirmed): a session is a maximal run
#         of events from the SAME client_ip + user_agent where each
#         event follows the previous one within 30 minutes (strictly
#         more than 30:00 apart starts a new session). user_agent is
#         the device identity; NULL user_agent gets a hidden
#         '(null-ua)' sentinel for grouping but is STORED raw NULL
#         (those 1,690 sessions join dim_device to nothing and surface
#         'unknown', exactly like null-UA fact rows). Ties on
#         event_time break on log_id, so the computation is
#         deterministic. NULL client_ip rows are excluded
#         (defensive guard - silver has 0; a null IP has no geo and
#         no session identity).
# Grain:  exactly one row per session - derived from ALL silver
#         events at each run, unlike the copy-through facts. The
#         session MERGE key session_id = xxhash64(client_ip | ua_key
#         | session_start): deterministic, and 64-bit because the
#         project's crc32 convention COLLIDES at this cardinality -
#         pre-flight found exactly 1 crc32 collision across 110,848
#         session keys (a colliding MERGE key would silently merge
#         two distinct sessions, which is fatal), while xxhash64 has
#         0 (expected ~3e-10 at this size). Pre-flight baseline for
#         post-build verification: 110,848 sessions over all 538,152
#         events (every event in exactly one session), 29,630
#         authenticated / 81,218 anonymous, avg 4.85 events, 18,917
#         single-event sessions, avg duration 7.08 min, max 180 min.
# Joins:  client_ip -> dim_ip_geo (KPI 10 sessions by country),
#         user_agent -> dim_device (device split; 28,038 bot
#         sessions are RETAINED - the fact is gross, bot exclusion
#         stays a view policy like fact_web_logs), date_sk = smart
#         YYYYMMDD of session_start (range 20241029..20261106,
#         fully covered by dim_date).
# Columns: session start/end (ART wall clock, the silver event_time
#         clock), duration_seconds (end - start; 0 for
#         single-event sessions), event_count (requests in session -
#         the pages-per-session proxy), distinct_endpoints (unique
#         endpoints hit), entry_referer (FIRST event's referer, raw
#         URL - KPI 10 top traffic sources: google.com 17,457 /
#         bing 8,776 / linkedin 8,768 sessions; NULL = direct or
#         unknown - 54,498 sessions; domain derivation is a view
#         concern), customer_sk (conversion bridge: the session's
#         customer when EXACTLY ONE distinct customer appears -
#         29,630 sessions; NULL for anonymous sessions and for the
#         5 sessions that touch more than one distinct customer,
#         where a single attribution would be wrong).
# Late data: a late-arriving event that precedes a session start
#         shifts session_start, hence a NEW session_id - the AUTO
#         CDC snapshot diff then DELETES the old session row and
#         INSERTS the recomputed one. That is correct Type 1
#         behavior, not a duplicate: keys are derived, never
#         recycled.
# MERGE / idempotency: AUTO CDC FROM SNAPSHOT into a Delta-backed
#         streaming table keyed on session_id (requirements' MERGE
#         constraint); an unchanged source recomputes the identical
#         session set and diffs to zero events - retries change
#         nothing. Sessions fully removed from silver are deleted
#         here too.
# Quality: no expectations - AUTO CDC FROM SNAPSHOT flows cannot
#         carry them (same decision as every other gold table), and
#         silver owns row-level validation.
# Layout: partitioned by date_sk (requirements: fact_* partitioned
#         by date). No clustering keys - Delta forbids CLUSTER BY
#         alongside PARTITION BY (same decision as the other
#         facts); the date partitions prune MERGE scans and KPI
#         queries.
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_LOGS = "b2b_ecommerce.silver.logs"
GOLD = "b2b_ecommerce.gold"
NULL_UA = "(null-ua)"  # grouping sentinel only; user_agent is stored raw
GAP_SECONDS = 30 * 60  # user-confirmed 30-minute inactivity gap


# The complete session snapshot, recomputed at each pipeline run:
# every silver web event, customer-resolved, ordered per
# (client_ip, user_agent), flagged where the gap to the previous
# event exceeds 30 minutes, grouped into sessions. A batch temporary
# view - the engine reads it at each run (ingestion-time snapshot
# versioning). No driver-side actions.
@dp.temporary_view(name="fact_web_sessions_snapshot")
def fact_web_sessions_snapshot():
    # Same conversion bridge as fact_web_logs: auth_user ->
    # dim_customers.email (SCD1, unique - pre-flight verified 0
    # duplicates, all auth_users match exactly one customer).
    customers = (
        spark.read.table(f"{GOLD}.dim_customers")
        .where(F.col("email").isNotNull())
        .select("email", "customer_sk")
    )

    events = (
        spark.read.table(SILVER_LOGS)
        .where(F.col("client_ip").isNotNull())
        .join(customers, F.col("auth_user") == F.col("email"), "left")
        .select(
            "log_id",
            "client_ip",
            "event_time",
            "endpoint",
            "referer",
            "user_agent",
            "customer_sk",
            # NULL user_agent groups under the sentinel so those
            # events still sessionize; the raw NULL is stored below.
            F.coalesce("user_agent", F.lit(NULL_UA)).alias("ua_key"),
        )
    )

    # Deterministic per-identity event order (ties break on log_id).
    order_w = Window.partitionBy("client_ip", "ua_key").orderBy(
        "event_time", "log_id"
    )

    # A session starts at the first event of its identity, or where
    # the inactivity gap to the previous event EXCEEDS 30 minutes.
    prev_event_time = F.lag("event_time").over(order_w)
    flagged = events.withColumn(
        "new_session",
        F.when(
            prev_event_time.isNull()
            | (
                (F.col("event_time").cast("long") - prev_event_time.cast("long"))
                > F.lit(GAP_SECONDS)
            ),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )

    # Session number within the identity: running count of session
    # starts, in the same deterministic event order.
    sequenced = flagged.withColumn(
        "session_seq",
        F.sum("new_session").over(
            order_w.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        ),
    )

    sessions = (
        sequenced.groupBy("client_ip", "ua_key", "session_seq")
        .agg(
            # raw user_agent is constant inside a ua_key group
            # (NULL stays NULL) - min() just lifts it out
            F.min("user_agent").alias("user_agent"),
            F.min("event_time").alias("session_start"),
            F.max("event_time").alias("session_end"),
            F.count(F.lit(1)).alias("event_count"),
            F.count_distinct("endpoint").alias("distinct_endpoints"),
            F.count_distinct("customer_sk").alias("customer_count"),
            F.max("customer_sk").alias("any_customer_sk"),
            # first event's referer in event order (struct tie-break
            # on log_id keeps it deterministic)
            F.min_by("referer", F.struct("event_time", "log_id")).alias(
                "entry_referer"
            ),
        )
    )

    return (
        sessions
        # 64-bit key: crc32 collides at this cardinality (1 collision
        # across 110,848 pre-flight keys; xxhash64: 0)
        .withColumn(
            "session_id",
            F.xxhash64(
                F.concat_ws(
                    "|", "client_ip", "ua_key", F.col("session_start").cast("string")
                )
            ),
        )
        # attribution only when exactly one distinct customer in the
        # session: anonymous (0) and multi-customer (5) stay NULL
        .withColumn(
            "customer_sk",
            F.when(F.col("customer_count") == F.lit(1), F.col("any_customer_sk")),
        )
        .withColumn(
            "date_sk",
            (
                F.year("session_start") * 10000
                + F.month("session_start") * 100
                + F.dayofmonth("session_start")
            ).cast("int"),
        )
        .withColumn(
            "duration_seconds",
            (F.unix_timestamp("session_end") - F.unix_timestamp("session_start")).cast(
                "int"
            ),
        )
        .select(
            "session_id",
            "date_sk",
            "client_ip",
            "user_agent",
            "customer_sk",
            "session_start",
            "session_end",
            "duration_seconds",
            "event_count",
            "distinct_endpoints",
            "entry_referer",
        )
    )


# Delta-backed streaming table: the AUTO CDC target. Partitioned by
# the date key of the session start (requirements: fact_*
# partitioned by date). No clustering keys: Delta forbids CLUSTER BY
# together with PARTITION BY.
dp.create_streaming_table(
    name="fact_web_sessions",
    comment=(
        "Kimball periodic-grain fact: one row per web session, derived from "
        "silver.logs by the user-confirmed rule - same client_ip + user_agent, "
        "split where the inactivity gap exceeds 30 minutes (deterministic "
        "event_time + log_id order). session_id = xxhash64(client_ip | ua_key | "
        "session_start) is the MERGE key; the project's crc32 convention is "
        "deliberately NOT used here because crc32 has exactly 1 collision "
        "across the 110,848 build-time session keys, while xxhash64 has 0 "
        "(a colliding MERGE key would silently merge two distinct sessions). "
        "110,848 sessions over all 538,152 events; date_sk = YYYYMMDD of "
        "session_start (20241029..20261106, fully covered by dim_date). "
        "customer_sk attributes a session only when exactly one distinct "
        "customer appears in it (29,630 sessions; NULL for anonymous and for "
        "the 5 multi-customer sessions). entry_referer is the first event's "
        "raw referer (NULL = direct/unknown). Bot sessions are retained (the "
        "fact is gross; exclusion is a view policy). Late events that precede "
        "a session start shift session_id - the AUTO CDC snapshot diff then "
        "deletes the superseded session and inserts the recomputed one. "
        "MERGE-loaded via AUTO CDC FROM SNAPSHOT keyed on session_id (Type 1, "
        "idempotent retries). Partitioned by date_sk (requirements: fact_* "
        "partitioned by date)."
    ),
    partition_cols=["date_sk"],
)


# The auto-merge load: diff each run's session snapshot against the
# last committed one and apply inserts/updates/deletes in place
# (Type 1 - facts do not version history; the dims own that).
dp.create_auto_cdc_from_snapshot_flow(
    target="fact_web_sessions",
    source="fact_web_sessions_snapshot",  # ingestion-time snapshot versioning
    keys=["session_id"],
    stored_as_scd_type=1,
)